## Apex Wealth Data Pipeline

In [2]:
!pip install request
!pip install python-dotenv

ERROR: Could not find a version that satisfies the requirement request (from versions: none)
ERROR: No matching distribution found for request

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\NED\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\NED\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [3]:
import requests
import pandas as pd 
from dotenv import load_dotenv
import os
import psycopg2
from sqlalchemy import create_engine

## Extraction

In [5]:
load_dotenv
api_key = os.getenv('API_KEY')

api_key

'0bb27de9424a464ba47cd89911e640de'

In [27]:
# define url endpoint
symbol = 'AAPL'
url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'

In [28]:
# make a request to the API 
response = requests.get(url)
response.status_code

200

In [29]:
# get data in json format
data = response.json()
data

{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-01-23 15:59:00',
   'open': '248.17000',
   'high': '248.21001',
   'low': '247.90500',
   'close': '247.97000',
   'volume': '496563'},
  {'datetime': '2026-01-23 15:58:00',
   'open': '248.039993',
   'high': '248.19000',
   'low': '248.029999',
   'close': '248.17500',
   'volume': '172096'},
  {'datetime': '2026-01-23 15:57:00',
   'open': '247.82001',
   'high': '248.10001',
   'low': '247.81000',
   'close': '248.039993',
   'volume': '168240'},
  {'datetime': '2026-01-23 15:56:00',
   'open': '247.97000',
   'high': '248',
   'low': '247.74500',
   'close': '247.81500',
   'volume': '128610'}],
 'status': 'ok'}

In [30]:
# tranfer data to a dataframe
def transform_data(data):
    
    
    # extract values from json
    time_series = data['values']
    
    #convert to dataframe
    df = pd.DataFrame(time_series)
    
    # convert columns to appropriate data type
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    df = df.astype({
        'open':'float',
        'high':'float',
        'low':'float',
        'close':'float',
        'volume':'int'
    })
    return df

In [31]:
df = transform_data(data)
df

,datetime,open,high,low,close,volume
0,2026-01-23 15:59:00,248.170000,248.21001,247.905000,247.970000,496563
1,2026-01-23 15:58:00,248.039993,248.19000,248.029999,248.175000,172096
2,2026-01-23 15:57:00,247.820010,248.10001,247.810000,248.039993,168240
3,2026-01-23 15:56:00,247.970000,248.00000,247.745000,247.815000,128610


### Extract from multiple symbols

In [33]:
symbols =['AAPL','MSFT','GOOGL','AMZN']
all_data = []

def fetch_data(symbol,api_key):
    url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'
    response = requests.get(url)
    response.raise_for_status() # raise an error for bad responses
    data = response.json()
    
    #check if the api response is ok
    if data['status'] != 'ok':
        print(f"error with {symbol}: {data['message']}")
    return data
    

In [34]:
symbols =['AAPL','MSFT','GOOGL','AMZN']

for symbol in symbols:
    data = fetch_data(symbol,api_key)
    df = transform_data(data)
    df['sym'] = symbol # add a column for the symbol
    all_data.append(df)

In [35]:
all_data

[             datetime        open       high         low       close  volume  \
 0 2026-01-23 15:59:00  248.170000  248.21001  247.905000  247.970000  496563   
 1 2026-01-23 15:58:00  248.039993  248.19000  248.029999  248.175000  172096   
 2 2026-01-23 15:57:00  247.820010  248.10001  247.810000  248.039993  168240   
 3 2026-01-23 15:56:00  247.970000  248.00000  247.745000  247.815000  128610   
 
     sym  
 0  AAPL  
 1  AAPL  
 2  AAPL  
 3  AAPL  ,
              datetime       open       high        low      close  volume  \
 0 2026-01-23 15:59:00  466.70999  466.79001  465.81000  465.92999  603069   
 1 2026-01-23 15:58:00  466.32001  466.79999  466.31000  466.70999  196047   
 2 2026-01-23 15:57:00  466.53500  466.73001  466.20999  466.35001  288406   
 3 2026-01-23 15:56:00  466.64001  466.68000  466.41000  466.53000  187794   
 
     sym  
 0  MSFT  
 1  MSFT  
 2  MSFT  
 3  MSFT  ,
              datetime       open        high        low      close  volume  \
 0 2026-01

In [36]:
# using concat - joining or stacking all the data together
all_df = pd.concat(all_data,ignore_index=True)

In [37]:
all_df

,datetime,open,high,low,close,volume,sym
0,2026-01-23 15:59:00,248.170000,248.210010,247.905000,247.970000,496563,AAPL
1,2026-01-23 15:58:00,248.039993,248.190000,248.029999,248.175000,172096,AAPL
2,2026-01-23 15:57:00,247.820010,248.100010,247.810000,248.039993,168240,AAPL
3,2026-01-23 15:56:00,247.970000,248.000000,247.745000,247.815000,128610,AAPL
4,2026-01-23 15:59:00,466.709990,466.790010,465.810000,465.929990,603069,MSFT
5,2026-01-23 15:58:00,466.320010,466.799990,466.310000,466.709990,196047,MSFT
6,2026-01-23 15:57:00,466.535000,466.730010,466.209990,466.350010,288406,MSFT
7,2026-01-23 15:56:00,466.640010,466.680000,466.410000,466.530000,187794,MSFT
8,2026-01-23 15:59:00,328.120000,328.200010,327.870000,327.920010,390130,GOOGL
9,2026-01-23 15:58:00,328.010010,328.130000,327.950010,328.130000,182955,GOOGL


### Loading

In [39]:
# database credentials
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_PORT = os.getenv('DB_PORT')

In [40]:
DB_NAME

'apex_db'

In [44]:
#create connection string with url
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# load data to data base 
all_df.to_sql('stockprices_data',engine,if_exists='append',index=False)

print('data loaded successfully')

data loaded successfully


In [45]:
# save to csv
all_df.to_csv('stockprice_data.csv',index=False)